# Matched 16S rRNA Amplicon and Shotgun Metagenomic Taxonomic and Functional Profiles from a Metabolically Stratified Japanese Adult Cohort Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² AI-Ready Microbiome Resource using the `mlcroissant` library.

### Dataset Source
The dataset FAIR² is provided via a Croissant schema URL and describes taxonomic and functional profiles from faecal shotgun metagenomic and 16S rRNA gene sequencing of 306 Japanese adults.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.7w36-htnb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata as a single object (not dict)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID (@id): {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema defines entities such as record sets, fields, and columns with unique `@id` identifiers. We list all record sets and their fields using `mlcroissant`.

In [ ]:
# List all record sets in the dataset with their @ids and field @ids
record_set_objs = dataset.metadata.record_sets

print("Record Sets in the dataset:")
record_set_ids = []
for rs in record_set_objs:
    print(f"- RecordSet name: {getattr(rs, 'name', 'N/A')}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    fields = getattr(rs, 'fields', [])
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - {getattr(f, 'name', 'N/A')}: {f.id}")
    else:
        print("  No fields listed.")
print(f"\nTotal record sets found: {len(record_set_ids)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here, we extract all data tables defined in the Croissant schema and make DataFrames for each, referencing entities by `@id`.

In [ ]:
# Extract data from all available record sets
dataframes = {}
# In case no record sets found, this block won't run.
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, Columns: {df.columns.tolist()}")
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# Select first available record set for demonstration
example_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if example_record_set_id:
    print(f"\nDataFrame preview for RecordSet @id: {example_record_set_id}")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section demonstrates:
- Filtering records with high values in a numeric field
- Normalization
- Grouping by categorical field

*Note: For demonstration, the numeric field and group field are chosen based on the DataFrame columns. Please replace with actual `@id`s as necessary*.

In [ ]:
# Example EDA on the first record set
if example_record_set_id:
    df = dataframes[example_record_set_id]
    print(f"Columns for EDA: {df.columns.tolist()}")
    # Try to select a numeric field and a group field by guessing from column names
    numeric_candidates = [col for col in df.columns if col.lower().startswith('alpha') or col.lower().endswith('abundance') or 'value' in col.lower() or 'index' in col.lower() or df[col].dtype in [float, int]]
    group_candidates = [col for col in df.columns if 'group' in col.lower() or 'status' in col.lower() or 'sex' in col.lower() or 'category' in col.lower()]

    numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]
    group_field_id = group_candidates[0] if group_candidates else df.columns[0]

    print(f"Using numeric field for filtering: {numeric_field_id}")
    print(f"Using group field for grouping: {group_field_id}")

    threshold = df[numeric_field_id].mean() + df[numeric_field_id].std() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else df
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        try:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        except Exception as e:
            print(f"Could not group by {group_field_id}: {e}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the numeric field and show boxplots by group if available.

In [ ]:
# Visualization of numeric field
if example_record_set_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8,4))
    plt.hist(df[numeric_field_id], bins=30, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group
    if group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.suptitle('')
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR² microbiome dataset using Croissant schema and `mlcroissant`
- Reviewed and referenced dataset entities by their `@id`s
- Extracted and previewed data from available record sets
- Performed basic EDA including filtering, normalization, and grouping using field `@id`s
- Visualized data distributions

This approach demonstrates reproducible and machine-actionable exploration using FAIR and Croissant standards. For more advanced analyses, refer to the specific field and column `@id`s provided in the dataset metadata and documentation.